# 10 NLP Policy Intelligence

Text analytics on country policy documents: keyword extraction, topic tagging, and policy signal scoring.

> Run `python run_pipeline.py` first.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
processed = ROOT / 'data' / 'processed'
predictions = ROOT / 'outputs' / 'predictions'

analytics = ROOT / 'data' / 'analytics'

docs = pd.read_csv(processed / 'nlp_policy_documents.csv')
keywords = pd.read_csv(analytics / 'nlp_keywords.csv')
topics = pd.read_csv(analytics / 'topic_distribution.csv')
country_signals = pd.read_csv(analytics / 'country_policy_signals.csv', index_col=0)

print(f"Documents: {len(docs)}")
print(f"Countries: {docs.country.nunique()}")
print(f"Topics tracked: {topics.topic.nunique()}")
print(f"Unique keywords extracted: {keywords.term.nunique()}")

## Policy support signal by country

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
docs_sorted = docs.sort_values('policy_support_signal', ascending=False)
bars = ax.bar(docs_sorted['country'], docs_sorted['policy_support_signal'],
              color=plt.cm.RdYlGn(docs_sorted['policy_support_signal'] / 100))
ax.set_title('Policy Support Signal by Country (0–100)', fontsize=13)
ax.set_ylabel('Signal score')
ax.set_xlabel('')
ax.axhline(50, color='gray', linestyle='--', alpha=0.5, label='Neutral baseline')
ax.legend()
plt.tight_layout(); plt.show()

print("\nSignal breakdown:")
print(docs[['country','policy_support_signal','detected_topics']].sort_values('policy_support_signal', ascending=False).to_string(index=False))

## Topic presence heatmap by country

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
sns.heatmap(country_signals, annot=True, fmt='.0f', cmap='YlOrRd',
            cbar_kws={'label': 'Topic present (1/0)'}, ax=ax)
ax.set_title('Policy Topic Presence by Country', fontsize=13)
ax.set_xlabel('Topic'); ax.set_ylabel('Country')
plt.tight_layout(); plt.show()

## Top keywords by country

In [ ]:
top_kw = keywords.groupby(['country','term'], as_index=False)['count'].sum().sort_values('count', ascending=False)

for country in docs['country'].unique():
    kw = top_kw[top_kw.country == country].head(8)
    print(f"\n{country}: {', '.join(kw['term'].tolist())}")

## Global keyword frequency

In [ ]:
global_kw = keywords.groupby('term', as_index=False)['count'].sum().sort_values('count', ascending=False).head(25)

fig, ax = plt.subplots(figsize=(12, 5))
sns.barplot(data=global_kw, x='term', y='count', ax=ax, palette='Blues_r')
ax.set_title('Top 25 Keywords Across All Policy Documents', fontsize=13)
ax.set_xlabel(''); ax.tick_params(axis='x', rotation=45)
plt.tight_layout(); plt.show()

## How the topic tagging works

In [ ]:
from src.nlp.topic_tagging import TOPICS

print("Topic definitions (keyword lists):")
for topic, keywords_list in TOPICS.items():
    print(f"  {topic:<20}: {', '.join(keywords_list)}")

## Limitations and what a production NLP layer would look like

In [ ]:
print("""
Current implementation:
- Rule-based keyword matching (deterministic, auditable, no training data needed)
- Works on curated policy document samples
- Produces binary topic presence flags + a simple additive score

Production improvements worth adding:
1. Sentence-level context: 'no SMR plans' should NOT trigger SMR signal
2. Sentiment: distinguish pro-nuclear vs anti-nuclear framing
3. Named entity recognition: extract specific projects, vendors, capacities
4. Document volume weighting: a 50-page policy document ≠ a 2-page press release
5. Temporal tracking: how has a country's policy stance changed over time?
6. LLM summarization: structured extraction of key commitments and timelines

For this portfolio, the rule-based approach is intentional — it's transparent,
auditable, and doesn't require training data the project doesn't have.
""")